# Master Notebook — Credit Default Models
Each model lives in `models/` and exposes `run(data)` (training data only) and `report(res)` (cross-validated results).
All models share the same train/test split, 5-fold stratified CV and cost matrix (bad classed good = 5, good classed bad = 1).

**Workflow**
1. *Model development* — run, compare and iterate using **CV results only**.
2. *Final evaluation* — once every model is frozen, run that section **once**. Do not go back and tweak models after seeing test results.

To add a model: copy `models/template.py`, fill in `build()` and `GRID`, then add a cell below.

In [ ]:
%matplotlib inline
from features import get_data
from models import common

data = get_data()
results = {}
print({k: v.shape for k, v in data.items()})

# Part 1 — Model development (training data / CV only)

## Logistic Regression

In [ ]:
from models import logistic_regression as lr

results["lr"] = lr.run(data)
lr.report(results["lr"])

## Random Forest

In [ ]:
# from models import random_forest as rf
# results["rf"] = rf.run(data)
# rf.report(results["rf"])

## XGBoost

In [ ]:
from models import xgboost_model as xgbm

results["xgb"] = xgbm.run(data)
xgbm.report(results["xgb"])

## Model selection (CV)
Choose models on these cross-validated numbers, not on test results.

In [ ]:
common.compare(results.values(), on="cv")

# Part 2 — Final evaluation (run ONCE, after all models are frozen)
Everything below uses the held-out test set.

In [ ]:
for res in results.values():
    common.evaluate(res, data)
    common.report(res, on="test")

common.compare(results.values(), on="test")

## Explainability (SHAP)

In [ ]:
sv = common.explain(results["lr"], data)

## Fairness analysis

In [ ]:
fair = common.fairness(results["lr"], data)